# Mini LCNN model


**The Final LCNN Architecture (No Optimisations) will have the following layers--Lavrantyeva et. al(2019)**

| Layer | Type      | Filter / Stride | Output Size | Params      |
| ----- | --------- | --------------- | ----------- | ----------- |
| 1     | Conv      | 5×5 / 1×1       | 863×600×64  | 1.6K        |
| 2     | MFM       | -               | 864×600×32  | -           |
| 3     | MaxPool   | 2×2 / 2×2       | 431×300×32  | -           |
| 4     | Conv      | 1×1 / 1×1       | 431×300×64  | 2.1K        |
| 5     | MFM       | -               | 431×300×32  | -           |
| 6     | BatchNorm | -               | 431×300×32  | -           |
| 7     | Conv      | 3×3 / 1×1       | 431×300×96  | 27.7K       |
| 8     | MFM       | -               | 431×300×48  | -           |
| 9     | MaxPool   | 2×2 / 2×2       | 215×150×48  | -           |
| 10    | BatchNorm | -               | 215×150×48  | -           |
| 11    | Conv      | 1×1 / 1×1       | 215×150×96  | 4.7K        |
| 12    | MFM       | -               | 215×150×48  | -           |
| 13    | BatchNorm | -               | 215×150×48  | -           |
| 14    | Conv      | 3×3 / 1×1       | 215×150×128 | 55.4K       |
| 15    | MFM       | -               | 215×150×64  | -           |
| 16    | MaxPool   | 2×2 / 2×2       | 107×75×64   | -           |
| 17    | Conv      | 1×1 / 1×1       | 107×75×128  | 8.3K        |
| 18    | MFM       | -               | 107×75×64   | -           |
| 19    | BatchNorm | -               | 107×75×64   | -           |
| 20    | Conv      | 3×3 / 1×1       | 107×75×64   | 36.9K       |
| 21    | MFM       | -               | 107×75×32   | -           |
| 22    | BatchNorm | -               | 107×75×32   | -           |
| 23    | Conv      | 1×1 / 1×1       | 107×75×64   | 2.1K        |
| 24    | MFM       | -               | 107×75×32   | -           |
| 25    | BatchNorm | -               | 107×75×32   | -           |
| 26    | Conv      | 3×3 / 1×1       | 107×75×64   | 18.5K       |
| 27    | MFM       | -               | 107×75×32   | -           |
| 28    | MaxPool   | 2×2 / 2×2       | 53×37×32    | -           |
| 29    | FC        | -               | 160         | 10.2M       |
| 30    | MFM       | -               | 80          | -           |
| 31    | BatchNorm | -               | 80          | -           |
| 32    | FC        | -               | 2           | 64          |
|       | **Total** |                 |             | **\~10.2M** |

---



In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

2025-07-28 20:34:34.036856: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-28 20:34:34.057127: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-28 20:34:34.241184: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-28 20:34:34.364198: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753727674.514812    2214 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753727674.55

## 1. Audio Preprocessing

```Feature extraction step-> takes in an audio file path and returns a spectrogram of the signal```

In [6]:
def get_spectrogram(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=16000)  # Load audio file
        
        # Create a Mel Spectrogram
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        
        # Convert to log scale (dB) for better feature representation
        log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
        
        return log_mel_spectrogram
    
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

## 2. LCNN Architecture(simple)

In [7]:
def build_lcnn(input_shape):
    model= Sequential([
        Input(shape=input_shape),
        
        #1st Conv. Block
        Conv2D(32,kernel_size=(3,3), activation='relu'),
        MaxPooling2D(pool_size=(2,2)),
        
        #2nd Conv. Block
        Conv2D(64,kernel_size=(3,3), activation='relu'),
        MaxPooling2D(pool_size=(2,2)),
        
        #3rd Conv. Block
        Conv2D(128,kernel_size=(3,3), activation='relu'),
        MaxPooling2D(pool_size=(2,2)),
        
        #flatten features to feed into dense layers
        Flatten(),
        
        #dense layers for classification
        Dense(128,activation='relu'),
        Dropout(0.5), #for overfitting
        
        # Output: 1 neuron with a sigmoid fucntion
        # (0 for bona fide, 1 for spoofed) 
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimiser='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'] )# accuracy is used for now during training, but eer will be used to evaluate the model after predictions
    
    return model
    

```Standard binary_crossentropy was used for this model (Softmax). Further optimisations of this model will help for Experiment 3, which focuses on determining the effect of training strategies on this model```

main()

In [ ]:
if __name__== '__main__':
    try:
        sample_audio_path = 'path/to/your/audio.wav'
        spectrogram = get_spectrogram(sample_audio_path)
        
        if spectrogram is not None:
            print("--- Spectrogram Generation ---")
            # Visualise the spectrogram
            plt.figure(figsize=(10, 4))
            librosa.display.specshow(spectrogram, sr=16000, x_axis='time', y_axis='mel')
            plt.colorbar(format='%+2.0f dB')
            plt.title('Mel-Spectrogram')
            plt.tight_layout()

            # build model and show summary
            # "channel" dimension is needed for the CNN
            spectrogram_shape = spectrogram.shape + (1,) 
            
            print("--- LCNN Model Architecture ---")
            poc_model = build_lcnn(input_shape=spectrogram_shape)
            poc_model.summary()

    except FileNotFoundError:
        print(f"\nThis file doesn't exist: '{sample_audio_path}'.")
    except Exception as e:
        print(f"An error occurred: {e}")